# Final Model Evaluation

This notebook performs the final comparison between the strongest
classical intent classifier and the DistilBERT classifier.

Models were selected using the training and validation splits before
this notebook.

The official Bitext test split has remained untouched throughout model
development and is evaluated here for the first time.

Models evaluated:

- Character TF-IDF + Linear SVM
- DistilBERT

Evaluation sources:

- Official leakage-safe Bitext test split
- Independent OOD challenge benchmark

No model tuning is performed in this notebook.

In [1]:
import numpy as np
import pandas as pd
import joblib
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

### Loading the Test Set

In [2]:
test_df = pd.read_csv(
    "../data/processed/generic/test.csv"
)

print(
    "Test shape:",
    test_df.shape
)

test_df.head()

Test shape: (4076, 9)


,flags,instruction,category,intent,response,normalized_instruction,split_group_key,processed_instruction,split
0,BL,I have got to cancel purchase {{Order Number}},ORDER,cancel_order,I'm sensitive to the fact that you're seeking ...,i have got to cancel purchase {{order number}},i have got to cancel purchase,I have got to cancel purchase ENTITY_TOKEN,test
1,BI,can you help me canceling order {{Order Number}}?,ORDER,cancel_order,I comprehend the urgency and complexity involv...,can you help me canceling order {{order number}}?,can you help me canceling order ?,can you help me canceling order ENTITY_TOKEN?,test
2,BLZ,help me canceling purchase {{Order Number}},ORDER,cancel_order,I catch on to the fact that you're seeking ass...,help me canceling purchase {{order number}},help me canceling purchase,help me canceling purchase ENTITY_TOKEN,test
3,BMQ,i have problems with canceling order {{Order N...,ORDER,cancel_order,I've been informed that you're facing some dif...,i have problems with canceling order {{order n...,i have problems with canceling order,i have problems with canceling order ENTITY_TOKEN,test
4,B,problem with cancelling order {{Order Number}},ORDER,cancel_order,I'm sensitive to the fact that you are experie...,problem with cancelling order {{order number}},problem with cancelling order,problem with cancelling order ENTITY_TOKEN,test


In [3]:
print(
    "Test intents:",
    test_df["intent"].nunique()
)

print(
    test_df[
        [
            "processed_instruction",
            "intent"
        ]
    ]
    .isna()
    .sum()
)

Test intents: 27
processed_instruction    0
intent                   0
dtype: int64


### Load Character-SVM

In [4]:
char_svm_pipeline = joblib.load(
    "../models/character_svm_pipeline.joblib"
)

In [5]:
X_test = test_df[
    "processed_instruction"
]

y_test = test_df[
    "intent"
]

char_test_predictions = (
    char_svm_pipeline.predict(
        X_test
    )
)

In [6]:
char_test_accuracy = accuracy_score(
    y_test,
    char_test_predictions
)

char_test_macro_f1 = f1_score(
    y_test,
    char_test_predictions,
    average="macro"
)

char_test_weighted_f1 = f1_score(
    y_test,
    char_test_predictions,
    average="weighted"
)

char_test_errors = (
    char_test_predictions
    != y_test
).sum()

print(
    f"Character-SVM Accuracy: "
    f"{char_test_accuracy:.4f}"
)

print(
    f"Character-SVM Macro F1: "
    f"{char_test_macro_f1:.4f}"
)

print(
    f"Character-SVM Weighted F1: "
    f"{char_test_weighted_f1:.4f}"
)

print(
    f"Character-SVM Errors: "
    f"{char_test_errors}/{len(test_df)}"
)

Character-SVM Accuracy: 0.9995
Character-SVM Macro F1: 0.9995
Character-SVM Weighted F1: 0.9995
Character-SVM Errors: 2/4076


### Load Saved Distil-BERT

In [7]:
DISTILBERT_PATH = (
    "../models/distilbert_intent_classifier/final"
)

distilbert_tokenizer = (
    AutoTokenizer.from_pretrained(
        DISTILBERT_PATH
    )
)

distilbert_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        DISTILBERT_PATH
    )
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [8]:
print(
    distilbert_model.config.num_labels
)

print(
    distilbert_model.config.id2label
)

27
{0: 'cancel_order', 1: 'change_order', 2: 'change_shipping_address', 3: 'check_cancellation_fee', 4: 'check_invoice', 5: 'check_payment_methods', 6: 'check_refund_policy', 7: 'complaint', 8: 'contact_customer_service', 9: 'contact_human_agent', 10: 'create_account', 11: 'delete_account', 12: 'delivery_options', 13: 'delivery_period', 14: 'edit_account', 15: 'get_invoice', 16: 'get_refund', 17: 'newsletter_subscription', 18: 'payment_issue', 19: 'place_order', 20: 'recover_password', 21: 'registration_problems', 22: 'review', 23: 'set_up_shipping_address', 24: 'switch_account', 25: 'track_order', 26: 'track_refund'}


### Convert Official Test Set for Distil-BERT

In [9]:
label2id = (
    distilbert_model.config.label2id
)

id2label = (
    distilbert_model.config.id2label
)

In [10]:
distilbert_test_df = (
    test_df[
        [
            "processed_instruction",
            "intent"
        ]
    ]
    .copy()
)

distilbert_test_df["label"] = (
    distilbert_test_df["intent"]
    .map(label2id)
)

assert (
    distilbert_test_df["label"]
    .notna()
    .all()
)

Create Data Set

In [11]:
test_dataset = Dataset.from_pandas(
    distilbert_test_df[
        [
            "processed_instruction",
            "label"
        ]
    ],
    preserve_index=False
)

Tokenization

In [12]:
MAX_LENGTH = 64

def tokenize_batch(batch):

    return distilbert_tokenizer(
        batch["processed_instruction"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [13]:
tokenized_test = (
    test_dataset.map(
        tokenize_batch,
        batched=True
    )
)

Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

### Create Evaluation Only Trainer

In [14]:
test_data_collator = (
    DataCollatorWithPadding(
        tokenizer=distilbert_tokenizer
    )
)

In [15]:
evaluation_args = TrainingArguments(
    output_dir="../models/final_evaluation_temp",
    per_device_eval_batch_size=16,
    report_to="none",
    dataloader_pin_memory=False
)

In [16]:
evaluation_trainer = Trainer(
    model=distilbert_model,
    args=evaluation_args,
    processing_class=distilbert_tokenizer,
    data_collator=test_data_collator
)

### Predict on Official Test Set

In [17]:
distilbert_test_output = (
    evaluation_trainer.predict(
        tokenized_test
    )
)

distilbert_test_predictions_ids = (
    np.argmax(
        distilbert_test_output.predictions,
        axis=-1
    )
)

In [18]:
distilbert_test_predictions = [
    id2label[int(prediction)]
    for prediction
    in distilbert_test_predictions_ids
]

In [19]:
distilbert_test_accuracy = (
    accuracy_score(
        y_test,
        distilbert_test_predictions
    )
)

distilbert_test_macro_f1 = (
    f1_score(
        y_test,
        distilbert_test_predictions,
        average="macro"
    )
)

distilbert_test_weighted_f1 = (
    f1_score(
        y_test,
        distilbert_test_predictions,
        average="weighted"
    )
)

distilbert_test_errors = (
    np.array(
        distilbert_test_predictions
    )
    != y_test.to_numpy()
).sum()

In [20]:
print(
    f"DistilBERT Accuracy: "
    f"{distilbert_test_accuracy:.4f}"
)

print(
    f"DistilBERT Macro F1: "
    f"{distilbert_test_macro_f1:.4f}"
)

print(
    f"DistilBERT Weighted F1: "
    f"{distilbert_test_weighted_f1:.4f}"
)

print(
    f"DistilBERT Errors: "
    f"{distilbert_test_errors}/{len(test_df)}"
)

DistilBERT Accuracy: 0.9953
DistilBERT Macro F1: 0.9953
DistilBERT Weighted F1: 0.9954
DistilBERT Errors: 19/4076


### Build the official test comparision

In [21]:
official_test_comparison = pd.DataFrame([
    {
        "model": "Character TF-IDF + Linear SVM",
        "accuracy": char_test_accuracy,
        "macro_f1": char_test_macro_f1,
        "weighted_f1": char_test_weighted_f1,
        "errors": char_test_errors
    },
    {
        "model": "DistilBERT",
        "accuracy": distilbert_test_accuracy,
        "macro_f1": distilbert_test_macro_f1,
        "weighted_f1": distilbert_test_weighted_f1,
        "errors": distilbert_test_errors
    }
])

official_test_comparison

,model,accuracy,macro_f1,weighted_f1,errors
0,Character TF-IDF + Linear SVM,0.999509,0.999509,0.999509,2
1,DistilBERT,0.995339,0.995347,0.995351,19


In [22]:
final_results = pd.DataFrame([
    {
        "model": "Character TF-IDF + Linear SVM",
        "validation_accuracy": 0.999502,
        "test_accuracy": char_test_accuracy,
        "test_macro_f1": char_test_macro_f1,
        "test_errors": char_test_errors,
        "challenge_accuracy": 0.4407,
        "challenge_macro_f1": 0.4179,
        "challenge_errors": 151,
    },
    {
        "model": "DistilBERT",
        "validation_accuracy": 0.997261,
        "test_accuracy": distilbert_test_accuracy,
        "test_macro_f1": distilbert_test_macro_f1,
        "test_errors": distilbert_test_errors,
        "challenge_accuracy": 0.5333,
        "challenge_macro_f1": 0.5180,
        "challenge_errors": 126,
    }
])

final_results

,model,validation_accuracy,test_accuracy,test_macro_f1,test_errors,challenge_accuracy,challenge_macro_f1,challenge_errors
0,Character TF-IDF + Linear SVM,0.999502,0.999509,0.999509,2,0.4407,0.4179,151
1,DistilBERT,0.997261,0.995339,0.995347,19,0.5333,0.5180,126


## Final Intent Classification Results

The final evaluation reveals a clear trade-off between
in-distribution performance and out-of-distribution generalization.

### In-Distribution Performance

Character TF-IDF + Linear SVM achieved:

- 99.95% validation accuracy
- 99.95% official test accuracy
- only 2 errors across 4,076 test examples

DistilBERT achieved:

- 99.73% validation accuracy
- 99.53% official test accuracy
- 19 errors across 4,076 test examples

These results confirm that character-level TF-IDF is exceptionally
well suited to the lexical and structural patterns present in the
Bitext dataset.

### Out-of-Distribution Performance

Performance changed substantially on the independently constructed
OOD challenge benchmark:

- Character TF-IDF + Linear SVM: 44.07% accuracy
- DistilBERT: 53.33% accuracy

DistilBERT therefore improved OOD accuracy by approximately 9.26
percentage points and reduced errors from 151 to 126.

The largest DistilBERT improvements appeared on paraphrased,
long-context, indirect, colloquial, and terse customer queries,
suggesting stronger transfer across changes in wording and sentence
structure.

### Interpretation

Character TF-IDF + Linear SVM is the strongest model for the
in-distribution Bitext task.

DistilBERT provides better generalization when customer language
moves outside the training distribution.

However, neither model achieves sufficiently robust performance on
the OOD benchmark for production deployment. The remaining challenge
performance suggests that training-data diversity is a larger
limitation than model architecture alone.

No models or hyperparameters were modified based on the official
test-set or challenge-set results.